In [2]:
import json
import pandas as pd
import os

# Get the current directory (where your script is running)
base_dir = os.path.dirname('/Users/nowshad/Desktop/Assignment/Data-Engineering/data/')
#print(base_dir+'/')
# ---------------------------
# 1. Read events.ndjson and prepare csv file for analysis
# ---------------------------
# --------------------------- # 1. Read events.ndjson # --------------------------- 
input_file = os.path.join(base_dir, "events.ndjson")
output_file = os.path.join(base_dir, "events.csv")
rows = []
with open(input_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        line = line.strip()
        # Skip empty lines
        if not line:
            continue
        # Load each NDJSON object
        try:
            obj = json.loads(line)
            rows.append(obj)
        except json.JSONDecodeError as e:
            print(f"Invalid JSON on line {i}: {line}")
            raise e  # stop so you can fix the file

# Convert to DataFrame
df_events = pd.DataFrame(rows)
# Save CSV
df_events.to_csv(output_file, index=False)
print("CSV created at:", output_file)

CSV created at: /Users/nowshad/Desktop/Assignment/Data-Engineering/data/events.csv


In [17]:
# ---------------------------
# 2. Read marketing_spend.csv
# ---------------------------
csv_path = os.path.join(base_dir, "marketing_spend.csv")
df_marketing_spend = pd.read_csv(csv_path)
print(df_marketing_spend.head())

         date     channel    spend
0  2026-01-01      Google   374.78
1  2026-01-01    Facebook  1995.09
2  2026-01-01      TikTok  1832.75
3  2026-01-01  Influencer  2287.86
4  2026-01-02      Google  2468.02


In [31]:
# ---------------------------
# 3. Read subscriptions.json and prepare csv file for analysis
# ---------------------------
json_path = os.path.join(base_dir, "subscriptions.json")
# Read JSON array into DataFrame
df_subscriptions = pd.read_json(json_path)
# Save as CSV in same directory
csv_path = os.path.join(base_dir, "subscriptions.csv")
df_subscriptions.to_csv(csv_path, index=False)
print("CSV created:", csv_path)
print(df_subscriptions.head())

CSV created: /Users/nowshad/Desktop/Assignment/Data-Engineering/data/subscriptions.csv
  subscription_id user_id plan_id  price currency  start_date    end_date  \
0       s_u0003_1   u0003   basic   9.99      USD  2026-01-12  2026-01-24   
1       s_u0003_2   u0003     pro  19.99      USD  2026-01-17        None   
2       s_u0004_1   u0004     pro  19.99      USD  2026-01-13  2026-01-31   
3       s_u0006_1   u0006     pro  19.99      USD  2026-01-11        None   
4       s_u0008_1   u0008   basic   9.99      USD  2026-01-10        None   

     status                created_at  
0  canceled 2026-01-12 01:00:00+00:00  
1    active 2026-01-17 02:00:00+00:00  
2  canceled 2026-01-13 01:00:00+00:00  
3    active 2026-01-11 01:00:00+00:00  
4    active 2026-01-10 01:00:00+00:00  


In [19]:
# ---------------------------
# 4. Messy and evolving source data marketing_spend.csv
# ---------------------------
print(df_marketing_spend.head())
print(len(df_marketing_spend))
# Drop duplicate rows based on all three columns
df_marketing_spend = df_marketing_spend.drop_duplicates(subset=["date", "channel", "spend"])

print(df_marketing_spend.head())
print(len(df_marketing_spend))

# ---------------------------
# Detect missing days (gaps)
# ---------------------------
full_range = pd.date_range(df_marketing_spend["date"].min(), df_marketing_spend["date"].max(), freq="D")
# Find missing days
missing_days = full_range.difference(df_marketing_spend["date"].unique())
print("Missing days:")
print(missing_days)

# ---------------------------
# Detect negative spend rows
# ---------------------------
negative_spend = df_marketing_spend[df_marketing_spend["spend"] < 0]
print("Negative spend rows:")
print(negative_spend)
# Remove negative spend rows
df_marketing_spend = df_marketing_spend[df_marketing_spend["spend"] >= 0]
print('After Messy and evolving source data marketing_spend: ', len(df_marketing_spend))

         date     channel    spend
0  2026-01-01      Google   374.78
1  2026-01-01    Facebook  1995.09
2  2026-01-01      TikTok  1832.75
3  2026-01-01  Influencer  2287.86
4  2026-01-02      Google  2468.02
100
         date     channel    spend
0  2026-01-01      Google   374.78
1  2026-01-01    Facebook  1995.09
2  2026-01-01      TikTok  1832.75
3  2026-01-01  Influencer  2287.86
4  2026-01-02      Google  2468.02
98
Missing days:
DatetimeIndex([], dtype='datetime64[ns]', freq=None)
Negative spend rows:
          date channel  spend
99  2026-01-08  Google -500.0
After Messy and evolving source data marketing_spend:  97


In [33]:
# def detect_subscription_edge_cases(df):
#     # Ensure datetime types
#     df = df.copy()
#     df["start_date"] = pd.to_datetime(df["start_date"])
#     df["end_date"] = pd.to_datetime(df["end_date"])
#     df["created_at"] = pd.to_datetime(df["created_at"])

#     overlaps = []
#     reactivations = []

#     # Process per user
#     for user_id, group in df.groupby("user_id"):
#         g = group.sort_values("start_date")

#         # ---- Detect overlaps ----
#         for i in range(1, len(g)):
#             prev = g.iloc[i-1]
#             curr = g.iloc[i]

#             prev_end = prev["end_date"] if pd.notna(prev["end_date"]) else pd.Timestamp.max
#             curr_end = curr["end_date"] if pd.notna(curr["end_date"]) else pd.Timestamp.max

#             if curr["start_date"] <= prev_end:
#                 overlaps.append({
#                     "user_id": user_id,
#                     "sub_a": prev["subscription_id"],
#                     "sub_b": curr["subscription_id"]
#                 })

#         # ---- Detect reactivations ----
#         for i in range(1, len(g)):
#             prev = g.iloc[i-1]
#             curr = g.iloc[i]

#             if prev["status"] == "canceled" and pd.notna(prev["end_date"]):
#                 if curr["start_date"] > prev["end_date"]:
#                     reactivations.append({
#                         "user_id": user_id,
#                         "previous_sub": prev["subscription_id"],
#                         "new_sub": curr["subscription_id"]
#                     })

#     return {
#         "overlaps": overlaps,
#         "reactivations": reactivations
#     }

# ---------------------------
# 5. Messy and evolving source data subscriptions.csv
# ---------------------------
# print(df_subscriptions.head())
print(len(df_subscriptions))
# Drop duplicate rows based on all columns
df_subscriptions = df_subscriptions.drop_duplicates(subset=[
    "subscription_id", "user_id", "plan_id", "price", "currency", "start_date", "end_date", "status", "created_at"
])
# print(df_subscriptions.head())
print(len(df_subscriptions))

# ---------------------------
# Detect duplicate subscription_id
# ---------------------------
duplicate_sub_ids = df_subscriptions[df_subscriptions.duplicated(subset=["subscription_id"], keep=False)]
print("Duplicate subscription_id rows:")
print(duplicate_sub_ids)
df_subscriptions = (
    df_subscriptions.sort_values("created_at")
      .drop_duplicates(subset=["subscription_id"], keep="first")
)


# results = detect_subscription_edge_cases(df_subscriptions)
# print("Overlapping subscriptions:")
# print(results["overlaps"])

# print("\nReactivations:")
# print(results["reactivations"])


433
433
Duplicate subscription_id rows:
Empty DataFrame
Columns: [subscription_id, user_id, plan_id, price, currency, start_date, end_date, status, created_at]
Index: []


In [23]:
# ---------------------------
# 6. Messy and evolving source data df_events.csv
# ---------------------------
print(df_events.head())
print('Length before apply duplicate event_id and conflicting duplicates: ', len(df_events))
# ---------------------------------------------------------
# Identify duplicate event_id rows
# ---------------------------------------------------------
duplicate_events = df_events[df_events.duplicated(subset=["event_id"], keep=False)]
# ---------------------------------------------------------
# Identify conflicting duplicates
#    (same event_id but different payload fields)
# ---------------------------------------------------------
payload_cols = ["amount", "currency"]   # add more payload fields if needed
conflicting = (
    df_events.groupby("event_id")[payload_cols]
      .nunique()
      .reset_index()
)
# event_ids where any payload column has >1 unique value
conflicting_ids = conflicting[
    conflicting[payload_cols].apply(lambda row: any(row > 1), axis=1)
]["event_id"].tolist()
# ---------------------------------------------------------
# Remove conflicting event_ids entirely
# ---------------------------------------------------------
df_events = df_events[~df_events["event_id"].isin(conflicting_ids)]
# ---------------------------------------------------------
# Remove simple duplicates (keep first occurrence)
# ---------------------------------------------------------
df_events = df_events.drop_duplicates(subset=["event_id"], keep="first")
# ---------------------------------------------------------
#  Updated dataframe is now clean
# ---------------------------------------------------------
print("Cleaned dataframe:")
print(len(df_events))


# ---------------------------------------------------------
# Missing fields / nulls (e.g., user_id missing/null)
# ---------------------------------------------------------
# Check how many rows have missing user_id
missing_user_id = df_events[df_events["user_id"].isna()]
print("Rows with missing user_id:")
print(missing_user_id)
# Remove rows where user_id is null or empty string
df_events = df_events[
    df_events["user_id"].notna() & (df_events["user_id"].astype(str).str.strip() != "")
]
print("Updated dataframe after removing missing user_id:")
print(len(df_events))


  event_id user_id event_type             timestamp  schema_version amount  \
0    e7978   u0458      login  2026-01-17T14:57:08Z               2    NaN   
1    e2734   u0019      login   2026-01-06 23:49:57               2    NaN   
2    e6712   u0127   purchase  2026-01-14T17:45:00Z               2   9.99   
3    e7071   u0349  page_view  2026-01-15T17:37:08Z               2    NaN   
4   e11844   u0171      login  2026-01-26T14:29:11Z               2    NaN   

  currency  tax   page refers_to_event_id acquisition_channel  
0      NaN  NaN    NaN                NaN                 NaN  
1      NaN  NaN    NaN                NaN                 NaN  
2      EUR  0.5    NaN                NaN                 NaN  
3      NaN  NaN  /home                NaN                 NaN  
4      NaN  NaN    NaN                NaN                 NaN  
Length before apply duplicate event_id and conflicting duplicates:  14052
Cleaned dataframe:
14052
Rows with missing user_id:
Empty DataFrame
Colum

In [34]:
# ---------------------------------------------------------
# Above raws completed below all points....
# ---------------------------------------------------------
# 1. Duplicate events (same event_id repeated)
# 2. Conflicting duplicates (same event_id, different payload such as amount/currency)
# 3. Out‑of‑order events (file order is not time order)
# 4. Late/early timestamps (including refunds that appear earlier than purchase timestamps)
# 5. Schema evolution (schema_version 1 and 2)
# 6. Inconsistent timestamp formats (ISO Z, ISO with offset, and some YYYY-MM-DD HH:MM:SS)
# 7. Corrupted rows (invalid JSON lines in the NDJSON)
# 8. Missing fields / nulls (e.g., user_id missing/null)
# 9. Marketing spend gaps (missing days) + duplicate rows + negative spend row
# 10. Subscriptions edge cases: overlaps, reactivations, and a duplicate subscription_id

In [66]:
# ---------------------------------------------------------
# ---------------------------------------------------------
# Business questions
# ---------------------------------------------------------
# ---------------------------------------------------------
# DAU: daily active users
# ---------------------------------------------------------

df_events["timestamp"] = pd.to_datetime(df_events["timestamp"], format="mixed", utc=True)
df_events["date"] = df_events["timestamp"].dt.date
dau = (
    df_events.groupby("date")["user_id"]
             .nunique()
             .reset_index(name="dau")
)
print("DAU: daily active users: ")
print(dau)

DAU: daily active users: 
          date  dau
0   2026-01-01  168
1   2026-01-02  190
2   2026-01-03  128
3   2026-01-04  132
4   2026-01-05  193
5   2026-01-06  218
6   2026-01-07  206
7   2026-01-08  145
8   2026-01-09  162
9   2026-01-10  207
10  2026-01-11  127
11  2026-01-12  191
12  2026-01-13  184
13  2026-01-14  185
14  2026-01-15  218
15  2026-01-16  163
16  2026-01-17  205
17  2026-01-18  214
18  2026-01-19  126
19  2026-01-20  135
20  2026-01-21  138
21  2026-01-22  138
22  2026-01-23  129
23  2026-01-24  191
24  2026-01-25  149
25  2026-01-26  175
26  2026-01-27  172
27  2026-01-28  208
28  2026-01-29  209
29  2026-01-30  196
30  2026-01-31    3
31  2026-02-01    1


In [67]:
# ---------------------------------------------------------
# Revenue: gross and net (refund‑adjusted) daily revenue
# ---------------------------------------------------------
purchase_events = df_events[df_events["event_type"] == "purchase"]
refund_events   = df_events[df_events["event_type"] == "refund"]
# Gross revenue
gross = (
    purchase_events.groupby("date")["amount"]
                   .sum()
                   .reset_index(name="gross_revenue")
)
# Refunds
refunds = (
    refund_events.groupby("date")["amount"]
                 .sum()
                 .reset_index(name="refund_amount")
)
# Merge
daily_revenue = gross.merge(refunds, on="date", how="left")
daily_revenue["refund_amount"] = daily_revenue["refund_amount"].fillna(0)
daily_revenue["net_revenue"] = daily_revenue["gross_revenue"] - daily_revenue["refund_amount"]
print("Revenue: gross and net (refund‑adjusted) daily revenue: ")
print(daily_revenue)

Revenue: gross and net (refund‑adjusted) daily revenue: 
          date gross_revenue  refund_amount net_revenue
0   2026-01-01        557.88          49.99      507.89
1   2026-01-02        755.84          99.00      656.84
2   2026-01-03        765.88          69.97      695.91
3   2026-01-04        706.91           0.00      706.91
4   2026-01-05        1062.9          99.98      962.92
5   2026-01-06         855.8         417.94      437.86
6   2026-01-07        666.81         624.94       41.87
7   2026-01-08        219.92          89.96      129.96
8   2026-01-09        279.88          49.99      229.89
9   2026-01-10        456.92           0.01      456.91
10  2026-01-11        903.91         178.97      724.94
11  2026-01-12        927.77         118.99      808.78
12  2026-01-13        378.88         109.97      268.91
13  2026-01-14        806.84         109.97      696.87
14  2026-01-15        923.83          79.97      843.86
15  2026-01-16       1052.89          39.98    

In [68]:
# ---------------------------------------------------------
# Weekly cohort retention: based on signup week
# ---------------------------------------------------------
signup = (df_events[df_events["event_type"] == "signup"].groupby("user_id")["date"].min().reset_index(name="signup_date"))
# Cohort week = Monday of signup week
signup["signup_week"] = signup["signup_date"] - signup["signup_date"].apply(lambda d: pd.Timestamp(d).weekday()) * pd.Timedelta(days=1)
# Merge into events
events2 = df_events.merge(signup, on="user_id", how="left")
# Event week
events2["event_week"] = events2["date"] - events2["date"].apply(lambda d: pd.Timestamp(d).weekday()) * pd.Timedelta(days=1)
# Cohort retention table
cohort = (events2.groupby(["signup_week", "event_week"])["user_id"].nunique().reset_index(name="active_users"))
print("Weekly cohort retention: based on signup week: ")
print(cohort)

Weekly cohort retention: based on signup week: 
  signup_week  event_week  active_users
0  2025-12-29  2025-12-29            35
1  2025-12-29  2026-01-05            33
2  2025-12-29  2026-01-12            34
3  2025-12-29  2026-01-19            30
4  2025-12-29  2026-01-26            30
5  2026-01-05  2025-12-29            78
6  2026-01-05  2026-01-05            92
7  2026-01-05  2026-01-12            91
8  2026-01-05  2026-01-19            86
9  2026-01-05  2026-01-26            80


/var/folders/1j/nn5d0__s1tn209qg1jhvwh8r0000gn/T/ipykernel_60196/3046973731.py:6: PerformanceWarning: Adding/subtracting object-dtype array to TimedeltaArray not vectorized.
  signup["signup_week"] = signup["signup_date"] - signup["signup_date"].apply(lambda d: pd.Timestamp(d).weekday()) * pd.Timedelta(days=1)
/var/folders/1j/nn5d0__s1tn209qg1jhvwh8r0000gn/T/ipykernel_60196/3046973731.py:10: PerformanceWarning: Adding/subtracting object-dtype array to TimedeltaArray not vectorized.
  events2["event_week"] = events2["date"] - events2["date"].apply(lambda d: pd.Timestamp(d).weekday()) * pd.Timedelta(days=1)


In [69]:
# ---------------------------------------------------------
# CAC: customer acquisition cost (paid conversions / spend)
# ---------------------------------------------------------
# ---  Normalize timestamps in events ---
df_events["timestamp"] = pd.to_datetime(df_events["timestamp"], errors="coerce", utc=True)
df_events["date"] = df_events["timestamp"].dt.date
# ---  Normalize dates in marketing spend ---
df_marketing_spend["date"] = pd.to_datetime(df_marketing_spend["date"]).dt.date
# ---  Paid conversions per day ---
paid_conv = (
    df_events[df_events["event_type"] == "purchase"]
    .groupby("date")["user_id"]
    .nunique()
    .reset_index(name="paid_conversions")
)
# ---  Merge with marketing spend ---
cac = df_marketing_spend.merge(paid_conv, on="date", how="left")
# Fill missing conversions with 0
cac["paid_conversions"] = cac["paid_conversions"].fillna(0)
# --- Compute CAC ---
cac["cac"] = cac.apply(
    lambda row: row["spend"] / row["paid_conversions"] if row["paid_conversions"] > 0 else None,
    axis=1
)
print("CAC: customer acquisition cost (paid conversions / spend): ")
print(cac)

CAC: customer acquisition cost (paid conversions / spend): 
          date     channel    spend  paid_conversions         cac
0   2026-01-01      Google   374.78                10   37.478000
1   2026-01-01    Facebook  1995.09                10  199.509000
2   2026-01-01      TikTok  1832.75                10  183.275000
3   2026-01-01  Influencer  2287.86                10  228.786000
4   2026-01-02      Google  2468.02                11  224.365455
..         ...         ...      ...               ...         ...
92  2026-01-29      TikTok  1790.02                17  105.295294
93  2026-01-29  Influencer  1942.03                17  114.237059
94  2026-01-30      Google  1468.28                17   86.369412
95  2026-01-30    Facebook  2311.88                17  135.992941
96  2026-01-30      TikTok   936.42                17   55.083529

[97 rows x 5 columns]


In [71]:
# ---------------------------------------------------------
# LTV: per user lifetime value (refund‑adjusted)
# ---------------------------------------------------------
user_purchases = (
    purchase_events.groupby("user_id")["amount"]
                   .sum()
                   .reset_index(name="gross_revenue")
)
# Refunds per user
user_refunds = (
    refund_events.groupby("user_id")["amount"]
                 .sum()
                 .reset_index(name="refund_amount")
)
ltv = user_purchases.merge(user_refunds, on="user_id", how="left")
ltv["refund_amount"] = ltv["refund_amount"].fillna(0)
ltv["ltv"] = ltv["gross_revenue"] - ltv["refund_amount"]
print("LTV: per user lifetime value (refund‑adjusted): ")
print(ltv)


# ---------------------------------------------------------
# LTV:CAC ratio
# ---------------------------------------------------------
total_ltv = ltv["ltv"].sum()
total_cac = df_marketing_spend["spend"].sum()
ltv_cac_ratio = total_ltv / total_cac
print("LTV:CAC ratio =", ltv_cac_ratio)

LTV: per user lifetime value (refund‑adjusted): 
    user_id gross_revenue  refund_amount     ltv
0     u0002         39.98           0.00   39.98
1     u0004        129.96          19.99  109.97
2     u0005         49.99           0.00   49.99
3     u0007          99.0           0.00    99.0
4     u0008        316.99           0.00  316.99
..      ...           ...            ...     ...
267   u0492         79.96           0.00   79.96
268   u0495          99.0           0.00    99.0
269   u0497          99.0           0.00    99.0
270   u0499          9.99           0.00    9.99
271   u0500         39.98          39.98     0.0

[272 rows x 4 columns]
LTV:CAC ratio = 0.14185070353677015


In [70]:
# ---------------------------------------------------------
# MRR: monthly recurring revenue (subscription‑based; not the same as revenue)
# ---------------------------------------------------------
df = df_subscriptions.copy()

# Convert dates
df["start_date"] = pd.to_datetime(df["start_date"])
df["end_date"] = pd.to_datetime(df["end_date"])
df["created_at"] = pd.to_datetime(df["created_at"])

# Replace null end_date with "today" (active subscriptions)
today = pd.Timestamp.today().normalize()
df["end_date_filled"] = df["end_date"].fillna(today)

# Build month range only from REAL subscription activity
start_month = df["start_date"].min().to_period("M").to_timestamp()
end_month = df["end_date_filled"].max().to_period("M").to_timestamp()

all_months = pd.date_range(start_month, end_month, freq="MS")

# Expand subscriptions across active months
rows = []

for _, row in df.iterrows():
    active_months = all_months[
        (all_months >= row["start_date"].to_period("M").to_timestamp()) &
        (all_months <= row["end_date_filled"].to_period("M").to_timestamp())
    ]
    
    for m in active_months:
        rows.append({
            "month": m,
            "subscription_id": row["subscription_id"],
            "user_id": row["user_id"],
            "plan_id": row["plan_id"],
            "price": row["price"]
        })

df_mrr = pd.DataFrame(rows)
# Compute MRR
mrr = (
    df_mrr.groupby("month")["price"]
          .sum()
          .reset_index(name="MRR")
)
print('3. MRR: monthly recurring revenue: ')
print(mrr)

3. MRR: monthly recurring revenue: 
       month      MRR
0 2026-01-01  8745.80
1 2026-02-01  6157.11
2 2026-03-01    19.99
